In [1]:
#import libraries
import numpy as np
import cartopy.crs as ccrss

import matplotlib.pyplot as plt

import xarray as xr

In [11]:
#Load in netcdf files

data_path = './../data/temporal/raw_data/'
#Manus Island (Tropical Location)
ds_manus = xr.open_dataset(data_path + 'manuscombined.nc') 
#Lamont, Ok (great plains location)
ds_plains = xr.open_dataset(data_path + 'wbpluvcombined.nc') 

In [12]:
# Identify bad days that contain NaNs
bad_dates_manus = ds_manus["time.date"].where(ds_manus["org_precip_rate_mean"].isnull(), drop=True).values
bad_dates_plains = ds_plains["time.date"].where(ds_plains["intensity_rt"].isnull(), drop=True).values

# Remove those days before grouping
filtered_manus = ds_manus.sel(time=~ds_manus["time.date"].isin(bad_dates_manus))
filtered_plains = ds_plains.sel(time=~ds_plains["time.date"].isin(bad_dates_plains))

# Group once and collect valid groups
valid_groups_manus = [group for _, group in filtered_manus.groupby("time.date")]
valid_groups_plains = [group for _, group in filtered_plains.groupby("time.date")]

In [13]:
#Concatenate valid groups
ds_manus_cleaned = xr.concat(valid_groups_manus, dim='time')
ds_plains_cleaned = xr.concat(valid_groups_plains, dim='time')

In [14]:
#compare before and after cleaning data
print('Before clean manus: ' + str(len(ds_manus.groupby('time.date'))))
print('After clean manus: ' + str(len(ds_manus_cleaned.groupby('time.date'))))
print(' ')
print('Before clean plains: ' + str(len(ds_plains.groupby('time.date'))))
print('After clean plains: ' + str(len(ds_plains_cleaned.groupby('time.date'))))

Before clean manus: 3535
After clean manus: 3515
 
Before clean plains: 2676
After clean plains: 1768


In [15]:
def temporal_resample(ds):
    """
    Converts precipitation rate to per-minute values and resamples to
    30-min, 1-hr, 3-hr, 6-hr, 12-hr, and daily mean rates.
    """
    #Setup precipitation rate data
    precip_rate = ds
    precip_rate = precip_rate.sortby('time')
    
    #Change given hourly rates to minute rates
    rate_min = precip_rate/(60)
    
    #Aggregate rates to larger time steps
    rate_30min = rate_min.resample(time = "30min").mean(dim='time')
    rate_1hr = rate_min.resample(time = "1h").mean(dim='time')
    rate_3hr = rate_min.resample(time = "3h").mean(dim='time')
    rate_6hr = rate_min.resample(time = "6h").mean(dim='time')
    rate_12hr = rate_min.resample(time = "12h").mean(dim='time')
    rate_day = rate_min.resample(time = "1d").mean(dim='time')
    
    return (rate_min, rate_30min, rate_1hr, rate_3hr, rate_6hr, rate_12hr, rate_day)
    
#Resample and return tuple of various rates
manus_rates = temporal_resample(ds_manus_cleaned.org_precip_rate_mean)
plains_rates = temporal_resample(ds_plains_cleaned.intensity_rt)

In [ ]:
import pickle

# Save both tuples to a single pickle file
with open("./../data/temporal/cleaned_input/resampled_rates.pkl", "wb") as f:
    pickle.dump({"manus": manus_rates, "plains": plains_rates}, f)